In [2]:
#!/usr/bin/env python
"""
pyCaverDock Workflow

This script implements a workflow where you supply a receptor PDB file and a ligand file.
It creates a working subdirectory (named after the receptor), moves the PDB there,
modifies the CAVER configuration on the fly (adjusting first_frame, last_frame, and time_sparsity),
runs CAVER on the copied PDB (using the workdir as the -pdb option), and then for each tunnel found,
starts a separate CaverDock run using pyCaverDock.

Dependencies:
  - pyCaverDock (which provides classes like CaverDock, Receptor, Ligand, load_tunnel, discretize_tunnel, etc.)
  - pandas, matplotlib, py3Dmol, MDAnalysis
  - openbabel (via pybel)
"""

import os
import shutil
import subprocess
import pandas as pd
import matplotlib.pyplot as plt
import py3Dmol
from openbabel import pybel
from openbabel import openbabel as ob

# Import pyCaverDock API
from pycaverdock import (
    CaverDock,
    Receptor,
    Ligand,
    load_tunnel,
    discretize_tunnel,
    Direction,
    TrajectoryType,
    EnergyProfilePlot,
    plot_results
)
from pycaverdock.experiment import convert_eprofile_analysis
from pycaverdock.utils import get_basename

def prepare_protein(input_file):
    """Convert a PDB receptor to PDBQT using OpenBabel."""
    if os.path.splitext(input_file)[1].lower() != ".pdbqt":
        base = os.path.splitext(input_file)[0]
        output_file = base + ".pdbqt"
        conv = ob.OBConversion()
        conv.SetInFormat("pdb")
        conv.SetOutFormat("pdbqt")
        conv.AddOption("r", ob.OBConversion.OUTOPTIONS)  # rigid output
        obmol = ob.OBMol()
        conv.ReadFile(obmol, input_file)
        obmol.CorrectForPH()  # optional
        obmol.AddHydrogens()
        conv.WriteFile(obmol, output_file)
        return output_file
    return input_file

def prepare_ligand(input_file):
    """Convert a ligand (SDF or MOL2) to PDBQT using OpenBabel."""
    ext = os.path.splitext(input_file)[1].lower()[1:]
    base = os.path.splitext(input_file)[0]
    output_file = base + ".pdbqt"
    mols = list(pybel.readfile(ext, input_file))
    if not mols:
        raise ValueError(f"No molecules found in {input_file}")
    mol = mols[0]
    mol.addh()
    mol.make3D()
    mol.calccharges(model='gasteiger')
    mol.write("pdbqt", output_file, overwrite=True)
    return output_file

class CaverDockWrapper:
    def __init__(self, caver_path=None, caverdock_path=None):
        """
        Initialize with paths to the CAVER and CaverDock executables.
        If not provided, they are assumed to be in subdirectories "caver" and "caverdock".
        """
        self.caver_path = caver_path if caver_path else os.path.join(os.getcwd(), "caver", "caver.jar")
        self.caverdock_path = caverdock_path if caverdock_path else os.path.join(os.getcwd(), "caverdock", "caverdock")
    
    def run_caver(self, receptor_file, out_dir="output", first_frame=1, last_frame=10, time_sparsity=1):
        """
        Run CAVER on the receptor PDB.
        This method:
          • Creates a working subdirectory named after the receptor’s base name.
          • Copies the receptor PDB into that directory.
          • Reads and modifies the default CAVER config file to set first_frame, last_frame, and time_sparsity.
          • Runs CAVER with -pdb set to the working directory.
        """
        os.makedirs(out_dir, exist_ok=True)
        receptor_base = os.path.splitext(os.path.basename(receptor_file))[0]
        work_dir = os.path.join(out_dir, receptor_base)
        os.makedirs(work_dir, exist_ok=True)
        
        # Copy the receptor file into work_dir
        receptor_copy = os.path.join(work_dir, os.path.basename(receptor_file))
        shutil.copy(receptor_file, receptor_copy)
        
        # Read the default config file (assumed to be "config_default.txt" in the same dir as caver.jar)
        default_config_path = os.path.join(os.path.dirname(self.caver_path), "config_default.txt")
        with open(default_config_path, "r") as f:
            config_lines = f.readlines()
        
        # Modify parameters: first_frame, last_frame, time_sparsity
        new_config_lines = []
        for line in config_lines:
            stripped = line.strip()
            if stripped.startswith("time_sparsity"):
                new_config_lines.append(f"time_sparsity {time_sparsity}\n")
            elif stripped.startswith("first_frame"):
                new_config_lines.append(f"first_frame {first_frame}\n")
            elif stripped.startswith("last_frame"):
                new_config_lines.append(f"last_frame {last_frame}\n")
            else:
                new_config_lines.append(line)
        
        # Save modified config file in work_dir
        new_config_path = os.path.join(work_dir, "config.txt")
        with open(new_config_path, "w") as f:
            f.writelines(new_config_lines)
        
        # Build the CAVER command; note that -pdb now points to the work_dir (which contains the receptor PDB)
        cmd = [
            "java", "-jar", self.caver_path,
            "-home", os.path.join(os.getcwd(), "caver"),
            "-pdb", work_dir,
            "-conf", new_config_path,
            "-out", work_dir
        ]
        print("Running CAVER:", " ".join(cmd))
        subprocess.check_call(cmd)
        
        # Assume tunnels are saved as "tunnels.pdb" in work_dir
        tunnels_file = os.path.join(work_dir, "tunnels.pdb")
        return tunnels_file, work_dir

    def run_caverdock(self, receptor_pdbqt, ligand_pdbqt, tunnel, work_dir, output_subdir, **kwargs):
        """
        Run a CaverDock analysis (via pyCaverDock) on a given tunnel.
        The tunnel parameter is expected to be a discretized tunnel (e.g., from discretize_tunnel).
        The output_subdir is created under work_dir for this tunnel’s results.
        """
        os.makedirs(output_subdir, exist_ok=True)
        # Instantiate the pyCaverDock API object
        cd_api = CaverDock()
        # Create a descriptive title using the receptor base name and tunnel index
        title = f"{get_basename(receptor_pdbqt)}_{os.path.basename(output_subdir)}"
        print(f"Running CaverDock for {title}")
        trajectory = cd_api.run(
            receptor=receptor_pdbqt,
            ligand=ligand_pdbqt,
            tunnel=tunnel,
            direction=Direction.OUT,
            trajectory_type=TrajectoryType.LOWERBOUND,
            mpi_processes=4,
            seed=42,
            exhaustiveness=8,
            workdir=output_subdir,
            title=title
        )
        return trajectory

    def parse_energy_profile(self, profile_file):
        df = pd.read_csv(profile_file, delim_whitespace=True, names=["coordinate", "energy"])
        return df

    def plot_energy_profile(self, energy_df, output_png):
        plt.figure(figsize=(8, 6))
        plt.plot(energy_df["coordinate"], energy_df["energy"], marker='o')
        plt.xlabel("Tunnel coordinate")
        plt.ylabel("Energy (kcal/mol)")
        plt.title("CaverDock Energy Profile")
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(output_png)
        plt.close()
        print(f"Energy profile plot saved to {output_png}")

    def visualize_tunnel(self, receptor_file, tunnel_file, output_html):
        view = py3Dmol.view(width=800, height=600)
        with open(receptor_file, "r") as f:
            pdb_data = f.read()
        view.addModel(pdb_data, "pdb")
        with open(tunnel_file, "r") as f:
            tunnel_data = f.read()
        view.addModel(tunnel_data, "pdb")
        view.setStyle({'model': 0}, {'cartoon': {'color': 'spectrum'}})
        view.setStyle({'model': 1}, {'stick': {'color': 'red', 'radius': 0.3}})
        view.zoomTo()
        html_str = view._make_html()
        with open(output_html, "w") as f:
            f.write(html_str)
        print(f"Interactive visualization HTML saved to {output_html}")

def main_workflow(receptor_pdb, ligand_file, first_frame=1, last_frame=1, time_sparsity=1):
    # Create a working subdirectory named after the receptor base name
    receptor_base = os.path.splitext(os.path.basename(receptor_pdb))[0]
    main_workdir = "work_dir"
    os.makedirs(main_workdir, exist_ok=True)
    
    # Prepare receptor and ligand using OpenBabel conversions
    receptor_pdbqt = prepare_protein(receptor_pdb)
    ligand_pdbqt = prepare_ligand(ligand_file)
    
    # Run CAVER:
    # This will copy the receptor PDB into a subdir and run CAVER with our modified config.
    tunnels_file, work_dir = CaverDockWrapper().run_caver(
        receptor_file=receptor_pdb,
        out_dir=main_workdir,
        first_frame=first_frame,
        last_frame=last_frame,
        time_sparsity=time_sparsity
    )
    
    # Load tunnels using pyCaverDock's load_tunnel.
    tunnels = load_tunnel(tunnels_file)
    if not isinstance(tunnels, list):
        tunnels = [tunnels]
    
    # For each tunnel, create a subdirectory and run a CaverDock run.
    for idx, tunnel in enumerate(tunnels, start=1):
        tunnel_dir = os.path.join(work_dir, f"tunnel_{idx}")
        os.makedirs(tunnel_dir, exist_ok=True)
        # Discretize the tunnel (delta = 0.3 Å as an example)
        disc_tunnel = discretize_tunnel(tunnel, delta=0.3)
        # Run CaverDock for this tunnel using pyCaverDock
        traj = CaverDockWrapper().run_caverdock(
            receptor_pdbqt=receptor_pdbqt,
            ligand_pdbqt=ligand_pdbqt,
            tunnel=disc_tunnel,
            work_dir=work_dir,
            output_subdir=tunnel_dir,
            direction="OUT",
            trajectory_type="LOWERBOUND",
            exhaustiveness=8
        )
        # Save energy profile from the trajectory
        profile_file = os.path.join(tunnel_dir, "profile.dat")
        traj.energy_profile.write_to_dat(profile_file)
        print(f"Energy profile for tunnel {idx} saved to {profile_file}")
        
        # Convert energy profile to DataFrame and save CSV
        energy_df = traj.energy_profile.to_dataframe()  # assuming such a method exists
        csv_file = os.path.join(tunnel_dir, "analysis.csv")
        energy_df.to_csv(csv_file, index=False)
        print(f"Energy analysis for tunnel {idx} saved to {csv_file}")
        
        # Plot energy profile and save PNG
        plot = EnergyProfilePlot(f"{receptor_base}_tunnel_{idx}_profile", traj.energy_profile, TrajectoryType.LOWERBOUND)
        png_file = os.path.join(tunnel_dir, "energy_profile.png")
        plot_results([plot], output=png_file, share_axes=True)
        print(f"Energy profile plot for tunnel {idx} saved to {png_file}")
    
    print("Workflow complete.")

if __name__ == "__main__":
    receptor_pdb = "1be0.pdb"    # Supply your receptor PDB file
    ligand_file = "EtOH.sdf"      # Supply your ligand file (SDF or MOL2)
    # Adjust first_frame, last_frame, and time_sparsity as desired.
    main_workflow(receptor_pdb, ligand_file, first_frame=1, last_frame=1, time_sparsity=1)


*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is 1be0.pdb)



Running CAVER: java -jar /home/erikna/whatcat/miner/caver/caver.jar -home /home/erikna/whatcat/miner/caver -pdb work_dir/1be0 -conf work_dir/1be0/config.txt -out work_dir/1be0
Caver computation started.
Merging all PDB files into one multimodel PDB file.
Settings loaded from:
work_dir/1be0/config.txt

Basic parameters:
starting_point_atom 1017 1026 1404
probe_radius 0.9
shell_radius 3.0
shell_depth 4.0
frame_weighting_coefficient 1.0
frame_clustering_threshold 1.0

*** Processing 1be0.pdb ***
0 redundant tunnels removed.
0 tunnels stored.

Loading tunnels from /home/erikna/whatcat/miner/work_dir/1be0/data/tunnels/tunnels_1be0.pdb.obj
Loaded 0 tunnels for 1be0.pdb.
0 tunnels successfully loaded.

Leaving only the cheapest tunnel per cluster per snapshot - 0 tunnels kept.
Statistical analysis of results started.
Saving summary information.
Saving tunnel characteristics.
Saving tunnel-lining residues and atoms.


Feb 25, 2025 8:06:36 PM caver.ui.Launcher cluster
Feb 25, 2025 8:06:36 PM caver.ui.Launcher run
SEVERE: Unexpected.
java.lang.NullPointerException: Cannot invoke "geometry.primitives.Point.distance(geometry.primitives.Point)" because "averageVoronoiOrigin" is null
	at caver.ui.Statistics.saveResidueClusterTouches(Statistics.java:475)
	at caver.ui.Launcher.saveStatistics(Launcher.java:1026)
	at caver.ui.Launcher.output(Launcher.java:813)
	at caver.ui.Launcher.compute(Launcher.java:1222)
	at caver.ui.Launcher.run(Launcher.java:1276)
	at caver.ui.Launcher.main(Launcher.java:1411)




Finished successfully.

-----------------------------------------------------------------------
 Thank you for using CAVER, please cite:

 Chovancova, E., Pavelka, A., Benes, P., Strnad, O., Brezovsky, J.,
 Kozlikova, B., Gora, A., Sustr, V., Klvana, M., Medek, P.,
 Biedermannova, L., Sochor, J. Damborsky, J. (2012) CAVER 3.0: A Tool
 for the Analysis of Transport Pathways in Dynamic Protein Structures.
 PLoS Comput Biol 8: e1002708. doi:10.1371/journal.pcbi.1002708
-----------------------------------------------------------------------


FileNotFoundError: Expected tunnel file at work_dir/1be0/tunnels.pdb